In [ ]:
# !pip install audio-separator[gpu] -q

In [ ]:
# import os
# import re
# import shutil
# from pathlib import Path

# import pandas as pd
# import torch
# from tqdm.auto import tqdm

# from audio_separator.separator import Separator
# from typing import Tuple
# import logging

# # =========================
# # CONFIG
# # =========================
# OUTPUT_DIR = Path("/kaggle/working/output")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# BG_DIR = OUTPUT_DIR / "background"
# VOCAL_DIR = OUTPUT_DIR / "vocals"
# VOCAL_DIR.mkdir(parents=True, exist_ok=True)
# BG_DIR.mkdir(parents=True, exist_ok=True)

# CSV_PATH = OUTPUT_DIR / "results.csv"
# ERROR_PATH = OUTPUT_DIR / "errors.csv"

# SEP_DIRS = {
#     "BS-RoFormer": Path("/kaggle/working/BS-RoFormer_seperator"),
#     "Mel-RoFormer": Path("/kaggle/working/Mel-RoFormer_seperator"),
#     "HTDemucs-FT": Path("/kaggle/working/HTDemucs-FT_seperator"),
# }

# separators = {
#     "BS-RoFormer": Separator(
#         output_dir=str(SEP_DIRS["BS-RoFormer"]),
#         output_format="wav",
#         log_level=logging.ERROR
#     ),
#     "Mel-RoFormer": Separator(
#         output_dir=str(SEP_DIRS["Mel-RoFormer"]),
#         output_format="wav",
#         log_level=logging.ERROR
#     ),
#     "HTDemucs-FT": Separator(
#         output_dir=str(SEP_DIRS["HTDemucs-FT"]),
#         output_format="wav",
#         log_level=logging.ERROR
#     ),
# }

# CACHE_DIRS = [
#     Path("/kaggle/input/datasets/dnnamm/vocal-bg/output/vocals"),
#     Path("/kaggle/input/datasets/dnnamm/vocals-bg-2/output/vocals"),
#     Path("/kaggle/input/datasets/nhatnam2610/vocals-bg-3/output/vocals")
# ]

# # =========================
# # HELPERS
# # =========================
# def extract_key(name: str) -> str:
#     """
#     Normalize filename để so khớp giữa vocal/background và audio gốc.
#     """
#     name = name.replace(".wav", "").replace(".mp3", "")
#     name = name.split("_(Vocals)")[0]
#     name = name.split("_(Instrumental)")[0]
#     name = name.strip().lower()
#     name = re.sub(r"^[._]+", "", name)   # bỏ prefix kiểu ._ .._
#     name = re.sub(r"_+", "_", name)      # gộp nhiều dấu _
#     return name


# def clear_sep_dir(model_name: str):
#     """
#     Xóa file tạm trong output_dir của separator trước mỗi lần chạy.
#     """
#     sep_dir = SEP_DIRS[model_name]
#     sep_dir.mkdir(parents=True, exist_ok=True)

#     for p in sep_dir.glob("*"):
#         if p.is_file():
#             try:
#                 p.unlink()
#             except Exception:
#                 pass


# def load(model_name="BS-RoFormer"):
#     if model_name == "BS-RoFormer":
#         separators["BS-RoFormer"].load_model("model_bs_roformer_ep_317_sdr_12.9755.ckpt")
#     elif model_name == "Mel-RoFormer":
#         separators["Mel-RoFormer"].load_model("model_mel_band_roformer_ep_3005_sdr_11.4360.ckpt")
#     elif model_name == "HTDemucs-FT":
#         separators["HTDemucs-FT"].load_model("htdemucs_ft.yaml")
#     else:
#         raise ValueError(
#             f'Not found model {model_name}. '
#             f'Please choose among: ["BS-RoFormer", "Mel-RoFormer", "HTDemucs-FT"]'
#         )


# def load_many_times(model_name, times=3):
#     last_err = None
#     for _ in range(times):
#         try:
#             load(model_name=model_name)
#             return
#         except Exception as e:
#             last_err = e
#             print(e)
#     raise RuntimeError(f"Cannot load model {model_name}: {last_err}")


# def merge(outs, model_name):
#     """
#     Gộp 3 stems của Demucs thành background.
#     """
#     import numpy as np
#     import soundfile as sf

#     waves = [sf.read(out)[0] for out in outs]
#     bgm = np.sum(np.array(waves), axis=0)

#     max_abs = max(abs(bgm.max()), abs(bgm.min()), 1e-8)
#     bgm = bgm / max_abs

#     tmp_file = os.path.join(
#         str(SEP_DIRS[model_name]),
#         f"{Path(outs[0]).stem}_merged.wav"
#     )
#     sf.write(tmp_file, bgm, 44100)
#     return tmp_file


# def separate(audio: str, model_name: str) -> Tuple[str, str]:
#     """
#     Return:
#         vocal_path, background_path
#     """
#     separator = separators[model_name]

#     outs = separator.separate(audio)
#     if not outs:
#         raise RuntimeError("Separator returned no output files")

#     outs = [os.path.join(str(SEP_DIRS[model_name]), out) for out in outs]

#     # RoFormer: 2 outputs
#     if len(outs) == 2:
#         return outs[1], outs[0]

#     # Demucs: 4 outputs
#     if len(outs) == 4:
#         bgm = merge(outs[:3], model_name)
#         return outs[3], bgm

#     raise RuntimeError(f"Unexpected output format: {len(outs)} files")


# def build_existing_vocal_cache(cache_dirs):
#     """
#     Cache dựa trên nhiều thư mục vocal.
#     Chỉ cần vocal tồn tại là coi như đã xử lý.
#     """
#     cache = set()
#     for cache_dir in cache_dirs:
#         cache_dir = Path(cache_dir)
#         for f in cache_dir.glob("*.wav"):
#             cache.add(extract_key(f.name))
#     return cache


# # =========================
# # MAIN PIPELINE
# # =========================
# def pipeline(input_dir_mp3: Path, input_dir_wav: Path, model_name="BS-RoFormer"):
#     input_dir_mp3 = Path(input_dir_mp3)
#     input_dir_wav = Path(input_dir_wav)

#     if not input_dir_mp3.exists():
#         raise FileNotFoundError(f"Không tồn tại thư mục input: {input_dir_mp3}")
#     if not input_dir_wav.exists():
#         raise FileNotFoundError(f"Không tồn tại thư mục input: {input_dir_wav}")

#     wav_files = {f.stem: f for f in input_dir_wav.glob("*.wav")}
#     mp3_files = {f.stem: f for f in input_dir_mp3.glob("*.mp3")}

#     all_stems = sorted(set(wav_files) | set(mp3_files))
#     if not all_stems:
#         raise ValueError("Không tìm thấy audio trong input folders")

#     audio_files = [wav_files.get(s, mp3_files.get(s)) for s in all_stems]
#     print(f"Found {len(audio_files)} audio files")

#     # load model 1 lần
#     load_many_times(model_name, times=2)

#     # cache đã xử lý từ nhiều folder
#     existing_vocals = build_existing_vocal_cache(CACHE_DIRS)
#     print(f"Found {len(existing_vocals)} already processed vocal keys")

#     rows = []
#     errors = []

#     # nạp CSV cũ nếu có
#     if CSV_PATH.exists():
#         try:
#             old_df = pd.read_csv(CSV_PATH)
#             rows.extend(old_df.to_dict(orient="records"))
#         except Exception:
#             pass

#     if ERROR_PATH.exists():
#         try:
#             old_err = pd.read_csv(ERROR_PATH)
#             errors.extend(old_err.to_dict(orient="records"))
#         except Exception:
#             pass

#     for audio_path in tqdm(audio_files, desc="Processing"):
#         key = extract_key(audio_path.name)

#         # skip nếu đã có vocal trong cache
#         if key in existing_vocals:
#             print(f'Skipped: {key}')
#             continue
            

#         try:
#             clear_sep_dir(model_name)

#             vocal, bg = separate(str(audio_path), model_name)

#             vocal_path_obj = Path(vocal)
#             bg_path_obj = Path(bg)

#             new_vocal = VOCAL_DIR / vocal_path_obj.name
#             shutil.move(str(vocal_path_obj), str(new_vocal))

#             new_bg = ""
#             if bg_path_obj.exists() and bg_path_obj.stat().st_size > 0:
#                 new_bg_obj = BG_DIR / bg_path_obj.name
#                 shutil.move(str(bg_path_obj), str(new_bg_obj))
#                 new_bg = str(new_bg_obj)

#             # update cache ngay
#             existing_vocals.add(key)

#             rows.append({
#                 "audio_name": audio_path.name,
#                 "audio_path": str(audio_path),
#                 "vocal_path": str(new_vocal),
#                 "background_path": new_bg,
#                 "status": "ok",
#                 "error": "",
#             })

#             if torch.cuda.is_available():
#                 torch.cuda.empty_cache()

#         except Exception as e:
#             print(f"❌ Error {audio_path.name}: {e}")

#             rows.append({
#                 "audio_name": audio_path.name,
#                 "audio_path": str(audio_path),
#                 "vocal_path": "",
#                 "background_path": "",
#                 "status": "failed",
#                 "error": str(e),
#             })

#             errors.append({
#                 "file": audio_path.name,
#                 "error": str(e),
#             })

#     df = pd.DataFrame(rows)
#     df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
#     print(f"✅ Saved CSV: {CSV_PATH}")

#     if errors:
#         err_df = pd.DataFrame(errors)
#         err_df.to_csv(ERROR_PATH, index=False, encoding="utf-8-sig")
#         print(f"❌ Saved error log: {ERROR_PATH}")

#     return df

In [ ]:
# df = pipeline(
#     input_dir_mp3=Path("/kaggle/input/datasets/dnnamm/mtik-audios"),
#     input_dir_wav=Path("/kaggle/input/datasets/dnnamm/mguard-audio-wav"),
#     model_name="BS-RoFormer"
# )

### Check audio after being processed

In [1]:
import re 
from pathlib import Path

def extract_key(name: str) -> str:
    name = name.replace(".wav", "").replace(".mp3", "")
    name = name.split("_(Vocals)")[0]
    name = name.split("_(Instrumental)")[0]

    # remove prefix ._ .__ .._
    name = re.sub(r"^[._]+", "", name)

    # gộp nhiều underscore
    name = re.sub(r"_+", "_", name)

    return name

In [22]:
vocal_dir = Path('/Users/nnam/Documents/Workspace/university/mguard/TikTok_Classification/datasets/splitted_audios/vocals')
downloaded_files = list(vocal_dir.glob("*.wav"))
downloaded_files = [extract_key(f.name) for f in downloaded_files]
print(f"Number of vocal files: {len(downloaded_files)}")

Number of vocal files: 4671


In [24]:
files = []
        
missing_files =[]
with open("/Users/nnam/Documents/Workspace/university/mguard/TikTok_Classification/datasets/splitted_audios/missing_files.txt", "a") as mf:
    for f in files:
        if f not in downloaded_files:
            missing_files.append(f)
            mf.write(f"{f}\n")
        
print(f"Tổng số file bị thiếu: {len(missing_files)}")


Tổng số file bị thiếu: 0


### Create mapping csv